# 04 — Ambitious experiments
Run only after notebooks 01–03 produce stable numbers.

**A.** Runtime/memory profiling — PatchTST P16/S8 vs no-patch ablation, Weather T=96.
**B.** Traffic T=96 (resource-permitting) — naive + PatchTST only.
**C.** Self-supervised masked patch pretraining on Weather, then fine-tune.
**D.** Transfer learning: pretrain on Electricity → fine-tune on Weather T=96.
**E.** Failure-case analysis — pick 3 representative test windows.


## Colab setup
Verify GPU, mount Drive, clone the repo, install requirements, and create the project tree on Drive.


In [ ]:
# 1. GPU check
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 2. Mount Drive (skipped automatically when not on Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/cs4782_patchtst_project'
    IN_COLAB = True
except Exception:
    PROJECT_ROOT = '.'
    IN_COLAB = False
print('PROJECT_ROOT =', PROJECT_ROOT, ' IN_COLAB =', IN_COLAB)


In [ ]:
# 3. Clone repo (Colab only). Update REPO_URL in scripts/build_notebooks.py
# and re-run that script to regenerate notebooks if the URL changes.
REPO_URL = 'https://github.com/Ash1R/ATSW64W-experiments.git'
if IN_COLAB:
    import os, subprocess
    os.chdir('/content')
    # Derive the clone directory from the URL's basename so it matches the repo name.
    REPO_DIRNAME = REPO_URL.rstrip('/').rsplit('/', 1)[-1]
    if REPO_DIRNAME.endswith('.git'):
        REPO_DIRNAME = REPO_DIRNAME[:-4]
    if not os.path.isdir(f'/content/{REPO_DIRNAME}'):
        # check=True so a bad URL fails loudly here instead of crashing the next chdir.
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIRNAME], check=True)
    os.chdir(f'/content/{REPO_DIRNAME}')
    subprocess.run(['git', 'pull'], check=False)
print('cwd =', __import__('os').getcwd())


In [ ]:
# 4. Install requirements (best-effort; resolved relative to the repo root
# regardless of where the kernel started, so headless `nbconvert` runs work).
import subprocess, sys, os
_req_dir = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_req_dir, 'requirements.txt')):
        break
    _req_dir = os.path.dirname(_req_dir)
_req = os.path.join(_req_dir, 'requirements.txt')
if os.path.isfile(_req):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', _req], check=False)
else:
    print('skipping pip install — requirements.txt not found from', os.getcwd())


In [ ]:
# 5. Make the project's `code/` directory importable.
# We add `code/` itself to sys.path (not the repo root) because the
# stdlib already ships a module named `code` that the IPython kernel
# imports before this cell runs — shadowing that cleanly is messy.
# This way every import is `from utils...`, `from data...`, `from models...`.
import sys, os
# When run with `jupyter nbconvert --execute`, the kernel's cwd is
# the notebook's directory (`notebooks/`), so locate the repo root
# by walking up until we find `code/`. On Colab we already chdir'd
# into the cloned repo above.
REPO_DIR = os.getcwd()
for _ in range(4):
    if os.path.isdir(os.path.join(REPO_DIR, 'code')):
        break
    REPO_DIR = os.path.dirname(REPO_DIR)
CODE_DIR = os.path.join(REPO_DIR, 'code')
if not os.path.isdir(CODE_DIR):
    raise RuntimeError(f'could not locate code/ from {os.getcwd()}')
os.chdir(REPO_DIR)
for p in (REPO_DIR, CODE_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)
print('REPO_DIR =', REPO_DIR)
from utils.colab import ensure_dirs
subdirs = ensure_dirs(PROJECT_ROOT)
for k, v in subdirs.items():
    print(f'{k:>12}  {v}')


## A. Runtime / memory profiling


In [ ]:
from run_experiment import load_config
from train import train_from_config
import os, json
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'results')

# Section A is profiling only — params / per-epoch time / peak memory, not
# convergence. The no_patch config (P=1,S=1) has 336 tokens per channel, so
# attention is O(N^2) and ~64x slower than patched. 1 epoch is enough to
# populate profile_*.json. DLinear was previously profiled here too but is
# now compared against via the PatchTST paper baselines (see nb05).
specs = [
    ('configs/weather_patchtst_96.yaml',          {'run_name': 'profile_patchtst_T96'}),
    ('configs/weather_ablation_no_patch_96.yaml', {'run_name': 'profile_nopatch_T96'}),
]
for path, ov in specs:
    cfg = load_config(path)
    cfg.update({'project_root': PROJECT_ROOT, 'output_dir': OUTPUT_DIR, 'epochs': 1, 'patience': 1})
    cfg.update(ov)
    train_from_config(cfg)


In [ ]:
import glob, json
# Skip stale profile_dlinear_*.json from previous runs — DLinear is no
# longer profiled locally; we cite the PatchTST paper for its numbers.
for f in sorted(glob.glob(os.path.join(OUTPUT_DIR, 'metrics', 'profile_*.json'))):
    if 'dlinear' in os.path.basename(f):
        continue
    s = json.load(open(f))
    print(f"{s['run_name']:>30}  params={s['num_params']:>9,}  "
          f"avg_epoch_s={s['avg_epoch_seconds']:6.2f}  "
          f"N={s.get('num_patches','-')}  N^2={s.get('attention_pairs','-')}  "
          f"peak_mb={s['gpu_memory_mb'].get('peak_alloc_mb','-')}")


## B. Traffic T=96


In [ ]:
import torch
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 14e9:
    for cfg_path, model_overrides in [
        ('configs/weather_patchtst_96.yaml', {'model': 'naive', 'dataset': 'traffic', 'run_name': 'traffic_naive_T96'}),
        ('configs/weather_patchtst_96.yaml', {'dataset': 'traffic', 'batch_size': 8, 'run_name': 'traffic_patchtst_T96'}),
    ]:
        cfg = load_config(cfg_path)
        cfg.update({'project_root': PROJECT_ROOT, 'output_dir': OUTPUT_DIR})
        cfg.update(model_overrides)
        train_from_config(cfg)
else:
    print('Skipping Traffic — needs >=16GB GPU.')


## C. Self-supervised masked patch pretraining
Mask 40% of non-overlapping patches and reconstruct them with MSE. After pretraining we save a checkpoint and use `weather_selfsupervised_96.yaml` to fine-tune.


In [ ]:
import torch, numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader
from data.dataset import build_data_bundle
from models.patchtst import PatchTST, _num_patches

SEQ_LEN, PATCH, STRIDE = 336, 16, 16
MASK_RATIO = 0.4
D_MODEL = 128

bundle = build_data_bundle(PROJECT_ROOT, 'weather', seq_len=SEQ_LEN, pred_len=96)
loader = DataLoader(bundle.train, batch_size=32, shuffle=True, drop_last=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ssl_model = PatchTST(seq_len=SEQ_LEN, pred_len=96, patch_len=PATCH, stride=STRIDE,
                     d_model=D_MODEL, n_heads=16, num_layers=3, d_ff=256, dropout=0.2).to(device)
recon_head = nn.Linear(D_MODEL, PATCH).to(device)
opt = torch.optim.Adam(list(ssl_model.parameters()) + list(recon_head.parameters()), lr=1e-4)

for epoch in range(20):
    ssl_model.train()
    total = 0.0; n = 0
    for x, _ in loader:
        x = x.to(device)
        x_t = x.transpose(1, 2)
        x_norm, mean, std = PatchTST._instance_norm(x_t)
        patches = ssl_model.embed(x_norm)
        pad = x_norm[..., -1:].expand(-1, -1, STRIDE)
        true_patches = torch.cat([x_norm, pad], -1).unfold(-1, PATCH, STRIDE)
        B, M, N, _ = patches.shape
        mask = (torch.rand(B, M, N, device=device) < MASK_RATIO).float()
        masked_patches = patches * (1 - mask).unsqueeze(-1)
        tokens = masked_patches.reshape(B * M, N, D_MODEL) + ssl_model.pos_embed
        encoded = ssl_model.encoder(ssl_model.encoder_dropout(tokens))
        recon = recon_head(encoded).reshape(B, M, N, PATCH)
        loss = (((recon - true_patches) ** 2) * mask.unsqueeze(-1)).sum() \
                / mask.sum().clamp(min=1) / PATCH
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item(); n += 1
    print(f'[ssl epoch {epoch}] loss={total/max(n,1):.4f}')

ckpt_path = os.path.join(OUTPUT_DIR, 'checkpoints', 'weather_patchtst_T96_pretrained_finetune.pt')
torch.save({
    'model': ssl_model.state_dict(),
    'optimizer': None, 'epoch': -1, 'best_val': float('inf'),
    'config': {'model': 'patchtst', 'dataset': 'weather', 'seq_len': SEQ_LEN, 'pred_len': 96,
               'patch_len': PATCH, 'stride': STRIDE, 'd_model': D_MODEL, 'n_heads': 16,
               'num_layers': 3, 'd_ff': 256, 'dropout': 0.2},
    'scaler_mean': bundle.scaler.mean, 'scaler_std': bundle.scaler.std,
}, ckpt_path)
print('saved', ckpt_path)


In [ ]:
from run_experiment import load_config
cfg = load_config('configs/weather_selfsupervised_96.yaml')
cfg.update({'project_root': PROJECT_ROOT, 'output_dir': OUTPUT_DIR, 'resume': True,
            'patch_len': 16, 'stride': 16})
train_from_config(cfg)


## D. Transfer learning
Caveat: M differs across datasets, so the head must be re-initialised on the target. Re-run the SSL loop with `dataset='electricity'` and a different ckpt name, then load only the encoder + patch embedding weights for Weather fine-tuning.


## E. Failure-case analysis


In [ ]:
import numpy as np, torch
from torch.utils.data import DataLoader
from data.dataset import build_data_bundle
from models.patchtst import PatchTST

ckpt_path = os.path.join(OUTPUT_DIR, 'checkpoints', 'weather_patchtst_L336_T96_P16S8_seed42.pt')
# weights_only=False because the checkpoint stores numpy arrays
# (scaler.mean/std) alongside the state_dict; PyTorch 2.6 changed
# the default to True. Safe here — we wrote the checkpoint ourselves.
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
cfg = ckpt['config']
bundle = build_data_bundle(PROJECT_ROOT, cfg['dataset'], cfg['seq_len'], cfg['pred_len'])
model = PatchTST(seq_len=cfg['seq_len'], pred_len=cfg['pred_len'], patch_len=cfg['patch_len'],
                 stride=cfg['stride'], d_model=cfg['d_model'], n_heads=cfg['n_heads'],
                 num_layers=cfg['num_layers'], d_ff=cfg['d_ff'], dropout=cfg['dropout'])
model.load_state_dict(ckpt['model']); model.train(False)

loader = DataLoader(bundle.test, batch_size=64, shuffle=False)
preds, trues, hists = [], [], []
with torch.no_grad():
    for x, y in loader:
        preds.append(model(x).numpy()); trues.append(y.numpy()); hists.append(x.numpy())
preds = np.concatenate(preds); trues = np.concatenate(trues); hists = np.concatenate(hists)
errs = ((preds - trues) ** 2).mean(axis=(1, 2))
good_idx = int(errs.argsort()[len(errs)//20])
med_idx = int(errs.argsort()[len(errs)//2])
bad_idx = int(errs.argsort()[-1])
print('selected indices:', good_idx, med_idx, bad_idx)

from utils.plotting import plot_prediction_sample
fig_dir = os.path.join(OUTPUT_DIR, 'figures')
for tag, idx in [('good', good_idx), ('median', med_idx), ('failure', bad_idx)]:
    plot_prediction_sample(
        bundle.scaler.inverse(hists[idx]),
        bundle.scaler.inverse(trues[idx]),
        bundle.scaler.inverse(preds[idx]),
        os.path.join(fig_dir, f'failure_case_{tag}.png'),
        title=f'Weather T=96 — {tag} (idx={idx})',
    )
print('wrote failure-case plots to', fig_dir)
